<a href="https://colab.research.google.com/github/MANI-WEBDEVE/Learn_AI/blob/main/machine_learning/Boosting/Gradient%E2%80%94Boosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np


X = np.array([20, 30, 40, 50])      # Age
y = np.array([4, 6, 5, 8])          # Kharcha
N = len(y)

print(np.full(N, np.mean(y)))

[5.75 5.75 5.75 5.75]


In [ ]:
import numpy as np

# ==========================
# Step 1: Data
# ==========================
X = np.array([20, 30, 40, 50])      # Age
y = np.array([4, 6, 5, 8])          # Kharcha
N = len(y)

print("Dataset:")
for i in range(N):
    print(f"Customer {chr(65+i)}: Age={X[i]}, Kharcha={y[i]}")
print("-" * 40)

# ==========================
# Step 2: Hyperparameters
# ==========================
T = 3               # Number of boosting rounds
nu = 1.0            # Learning rate (we'll use 1 for simplicity)

# ==========================
# Step 3: Initialize F0 = mean(y)
# ==========================
F = np.full(N, np.mean(y))  # F0(x) for all samples
print(f"Initial prediction F0 = mean(y) = {F[0]:.2f}")
print(f"Initial predictions: {F.round(2)}")
print(f"Actual y:            {y}")
print(f"Residuals (y - F0):  {(y - F).round(2)}")
print("=" * 50)

# Store trees for final prediction
trees = []

# ==========================
# Step 4: Boosting Rounds
# ==========================
for t in range(T):
    print(f"\n--- ROUND {t+1} ---")

    # Step 4.1: Compute residuals (negative gradient for MSE)
    residuals = y - F
    print(f"Residuals (r): {residuals.round(4)}")

    # Step 4.2: Find best decision stump (1 split on X)
    best_split = None
    best_left_val = None
    best_right_val = None
    best_mse = float('inf')

    # Try every possible threshold (between unique values)
    unique_vals = np.unique(X)
    for i in range(len(unique_vals) - 1):
        # Threshold = midpoint between two values
        threshold = (unique_vals[i] + unique_vals[i+1]) / 2

        # Split data
        left_mask = X <= threshold
        right_mask = X > threshold

        if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
            continue

        # Predict constant in each region = mean of residuals
        left_pred = np.mean(residuals[left_mask])
        right_pred = np.mean(residuals[right_mask])

        # Compute MSE of this split
        pred = np.where(left_mask, left_pred, right_pred)
        mse = np.mean((residuals - pred) ** 2)

        if mse < best_mse:
            best_mse = mse
            best_split = threshold
            best_left_val = left_pred
            best_right_val = right_pred
            best_pred = pred.copy()

    # If no split found (shouldn't happen here), use overall mean
    if best_split is None:
        best_split = X[0]
        best_left_val = np.mean(residuals)
        best_right_val = np.mean(residuals)
        best_pred = np.full(N, best_left_val)

    print(f"Best split: Age > {best_split}")
    print(f"Left prediction (≤{best_split}): {best_left_val:.4f}")
    print(f"Right prediction (> {best_split}): {best_right_val:.4f}")
    print(f"Tree predictions (h_t): {best_pred.round(4)}")

    # Step 4.3: Compute gamma (step size) for MSE
    numerator = np.sum(residuals * best_pred)
    denominator = np.sum(best_pred ** 2)

    if denominator == 0:
        gamma = 0.0
    else:
        gamma = numerator / denominator

    print(f"Gamma (γ_t): {gamma:.4f}")

    # Step 4.4: Update F
    F = F + nu * gamma * best_pred
    print(f"Updated F: {F.round(4)}")
    print(f"Actual y:  {y}")
    print(f"New residuals: {(y - F).round(4)}")

    # Save tree
    trees.append({
        'threshold': best_split,
        'left_val': best_left_val,
        'right_val': best_right_val,
        'gamma': gamma
    })

# ==========================
# Step 5: Final Results
# ==========================
print("\n" + "="*50)
print("FINAL RESULTS:")
print(f"Actual y:      {y}")
print(f"Predicted y:   {F.round(4)}")
print(f"Errors:        {(y - F).round(4)}")
print(f"Final MSE:     {np.mean((y - F)**2):.6f}")
print(f"Accuracy (R²): {1 - np.sum((y-F)**2)/np.sum((y-np.mean(y))**2):.4f}")

Dataset:
Customer A: Age=20, Kharcha=4
Customer B: Age=30, Kharcha=6
Customer C: Age=40, Kharcha=5
Customer D: Age=50, Kharcha=8
----------------------------------------
Initial prediction F0 = mean(y) = 5.75
Initial predictions: [5.75 5.75 5.75 5.75]
Actual y:            [4 6 5 8]
Residuals (y - F0):  [-1.75  0.25 -0.75  2.25]

--- ROUND 1 ---
Residuals (r): [-1.75  0.25 -0.75  2.25]
Best split: Age > 45.0
Left prediction (≤45.0): -0.7500
Right prediction (> 45.0): 2.2500
Tree predictions (h_t): [-0.75 -0.75 -0.75  2.25]
Gamma (γ_t): 1.0000
Updated F: [5. 5. 5. 8.]
Actual y:  [4 6 5 8]
New residuals: [-1.  1.  0.  0.]

--- ROUND 2 ---
Residuals (r): [-1.  1.  0.  0.]
Best split: Age > 25.0
Left prediction (≤25.0): -1.0000
Right prediction (> 25.0): 0.3333
Tree predictions (h_t): [-1.      0.3333  0.3333  0.3333]
Gamma (γ_t): 1.0000
Updated F: [4.     5.3333 5.3333 8.3333]
Actual y:  [4 6 5 8]
New residuals: [ 0.      0.6667 -0.3333 -0.3333]

--- ROUND 3 ---
Residuals (r): [ 0.      0.